# Image model fine-tune runner

Thin runner: clones the repo, installs deps, generates curriculum data, trains the LoRA, and saves outputs.

Settings: GPU T4 x2, internet ON. Outputs land in `/kaggle/working/outputs`.

In [ ]:
import os, sys, subprocess
os.environ['HF_HOME']='/tmp/hf_cache'; os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
subprocess.run([sys.executable,'-m','pip','install','--quiet','--upgrade','torchao'],check=True)
pk=['git+https://github.com/huggingface/diffusers.git','transformers>=5.17.0','accelerate','bitsandbytes','sentencepiece','protobuf','safetensors','Pillow','peft']
subprocess.run([sys.executable,'-m','pip','install','--quiet',*pk],check=True)
import torch; print('installed', torch.__version__, torch.cuda.get_device_name(0), flush=True)

In [ ]:
!git clone https://github.com/Mrun25/image-gen-finetune.git /kaggle/working/repo
%cd /kaggle/working/repo

In [ ]:
# Curriculum data: stage 1 (clean) + stage 2 (busier), 512x640, lettering-free
!python src/synth_posters.py --out /kaggle/working/data_s1 --n 350 --stage 1 --size 512 640 --seed 101
!python src/synth_posters.py --out /kaggle/working/data_s2 --n 350 --stage 2 --size 512 640 --seed 202

In [ ]:
# Two-phase LoRA training + base-vs-LoRA comparison sheet
!python src/train_lora.py --stage1 /kaggle/working/data_s1 --stage2 /kaggle/working/data_s2 \
    --steps1 250 --steps2 250 --height 512 --width 640 \
    --out /kaggle/working/outputs